In [42]:
from collections import defaultdict

import pandas as pd
import os
import json

from nlp_content_validity.data.read_data import read_dataset, read_validity

dataset='colquitt_et_al'
# val_metric='htd'
# nlp_metric='htd_nlp'
val_metric='htc'
nlp_metric='mean_focal'
basedir = f'../../data/interim/{dataset}'
models = list(map(lambda x: os.path.splitext(x)[0], filter(lambda x: x.endswith('json'), os.listdir(basedir))))
for val_metric, nlp_metric in [('htd', 'htd_nlp'), ('htc', 'mean_focal')]:
    for model in models:
        definitions, df_, focal_scales, orbiting_dict = read_dataset(dataset)
        relations = {(focal, scale2): label for focal, vals in orbiting_dict.items() for label, scale2 in vals.items()}
        for focal in focal_scales:
            relations[(focal, focal)] = 'focal'
        filename = f'{model}.json'
        model_name = os.path.splitext(filename)[0]
        print(f'processing {basedir}/{filename}')
        with open(f'{basedir}/{filename}', 'rb') as f:
            data = json.load(f)
        data_dict = defaultdict(dict)
        for i in data:
            for k, v in i.items():
                for kk, vv in v.items():
                    data_dict[k][relations[(k, kk)]] = vv

        similarities=list()
        for scale, vals in data_dict.items():
            df=pd.DataFrame(vals)
            df['scale']=scale
            df['item'] = df_[df_.scale==scale].item.values
            similarities.append(df)
        df = pd.concat(similarities)
        if val_metric == 'htd':
            df[f'item_{nlp_metric}'] = (2*df.focal)-df.orbiting_scale_1-df.orbiting_scale_2
        elif val_metric == 'htc':
            df[f'item_{nlp_metric}'] = df.focal
        else:
            raise NotImplementedError(f'metric {val_metric} is not implemented')
        # df[f'item_{nlp_metric}_01'] = (df[f'item_{nlp_metric}']-df[f'item_{nlp_metric}'].min())/(df[f'item_{nlp_metric}'].max()-df[f'item_{nlp_metric}'].min())
        validities = read_validity(dataset)
        df = pd.merge(df, validities, left_on='scale', right_on='focal_scale', how='left')

        nlp_metrics = pd.read_csv(f'../../data/processed/{dataset}/{model}.csv', index_col=0).rename(columns={'htd':"htd_nlp"})
        merged = pd.merge(validities, nlp_metrics, left_index=True, right_index=True)

        def scale01(col):
            return (col-col.min())/(col.max()-col.min())
        merged[f'{val_metric}_norm'] = scale01(merged[f'{val_metric}'])
        merged[f'{nlp_metric}_norm'] = scale01(merged[f'{nlp_metric}'])
        merged[f'{val_metric}_rank_diff'] = (merged[f'{val_metric}_norm'].rank()-merged[f'{nlp_metric}_norm'].rank())
        most_demoted = merged.sort_values(by=[f'{val_metric}_rank_diff'], ascending=False).head(10)[[f'{val_metric}_norm', f'{nlp_metric}_norm', f'{val_metric}_rank_diff']]
        most_promoted = merged.sort_values(by=[f'{val_metric}_rank_diff'], ascending=True).head(10)[[f'{val_metric}_norm', f'{nlp_metric}_norm', f'{val_metric}_rank_diff']]

        df['item_rank_diff'] = df[f'{val_metric}'].rank() - df[f'item_{nlp_metric}'].rank()
        most_inflated_items = df.sort_values(by=['item_rank_diff'], ascending=False)[['item', 'focal', 'orbiting_scale_1', 'orbiting_scale_2', 'scale',f'{val_metric}',  'item_rank_diff',]].head(10)
        most_deflated_items = df.sort_values(by=['item_rank_diff'], ascending=True)[['item', 'focal', 'orbiting_scale_1', 'orbiting_scale_2', 'scale',f'{val_metric}',  'item_rank_diff',]].head(10)
        negative_htd_items = df[df[f'item_{nlp_metric}']<0][['item', 'focal', 'orbiting_scale_1', 'orbiting_scale_2', 'scale',f'{val_metric}',  'item_rank_diff',]]
        out_dir='../../reports/error_analysis'
        with pd.ExcelWriter(f'{out_dir}/{model}_{val_metric}_error_analysis.xlsx') as writer:
            if val_metric == 'htd':
                negative_htd_items.to_excel(writer, sheet_name=f'negative_{nlp_metric}_items', index=False)
            most_deflated_items.to_excel(writer, sheet_name='most_inflated_items', index=False)
            most_inflated_items.to_excel(writer, sheet_name='most_deflated_items', index=False)
            most_promoted.to_excel(writer, sheet_name='most_promoted')
            most_demoted.to_excel(writer, sheet_name='most_demoted')

            merged.loc[~(merged.index.isin(most_demoted.index)|merged.index.isin(most_promoted.index))].sample(10)[[f'{val_metric}_norm', f'{nlp_metric}_norm', f'{val_metric}_rank_diff']].to_excel(writer, sheet_name='random')
            df[~(df.index.isin(most_deflated_items.index)|df.index.isin(most_inflated_items.index))].sample(10)[['item', 'focal', 'orbiting_scale_1', 'orbiting_scale_2', 'scale',val_metric,  'item_rank_diff',]].to_excel(writer, sheet_name='random_items', index=False)

processing ../../data/interim/colquitt_et_al/bow_lsa.json
processing ../../data/interim/colquitt_et_al/llm_gemini.json
processing ../../data/interim/colquitt_et_al/llm_mistral.json
processing ../../data/interim/colquitt_et_al/sentence_roberta.json
processing ../../data/interim/colquitt_et_al/sentence_t5.json
processing ../../data/interim/colquitt_et_al/task_nli_deberta_contradiction.json
processing ../../data/interim/colquitt_et_al/task_nli_deberta_entailment.json
processing ../../data/interim/colquitt_et_al/task_nli_deberta_neutral.json
processing ../../data/interim/colquitt_et_al/task_sts_cross_encoder.json
processing ../../data/interim/colquitt_et_al/word_ft_cosine.json
processing ../../data/interim/colquitt_et_al/word_ft_wmd.json
processing ../../data/interim/colquitt_et_al/word_glove_cosine.json
processing ../../data/interim/colquitt_et_al/word_glove_wmd.json
processing ../../data/interim/colquitt_et_al/word_w2v_cosine.json
processing ../../data/interim/colquitt_et_al/word_w2v_wmd